# Ejercicio: predecir el consumo de electricidad en España

Tenemos el consumo eléctrico de España **hora a hora** (2015-2018) y la temperatura en Madrid.
Vas a crear un modelo que **prediga la demanda eléctrica** (`demanda_mw`).

Columnas que vas a usar:

| Columna | Significado |
|---|---|
| `anio`, `mes`, `hora` | Año, mes (1-12) y hora (0-23) |
| `dia_semana` | 0 = lunes ... 6 = domingo |
| `fin_de_semana` | 1 si es sábado o domingo |
| `temp_madrid` | Temperatura en Madrid (°C) |
| `demanda_mw` | **Lo que queremos predecir** (MW) |

Rellena los huecos marcados con `___`. Cada ejercicio tiene una **pista**.

## Ejercicio 1. Cargar los datos
Sube `luz_espana_2015_2018.csv` a Colab. Carga el archivo y muestra las 5 primeras filas.

**Pista:** `pd.read_csv("nombre.csv")` y `.head()`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

datos = pd.read_csv("luz_espana_2015_2018.csv")
datos.head()

## Ejercicio 2. ¿Cuántas horas hay y cuál es la demanda media?

**Pista:** `len(datos)` y `datos["columna"].mean()`

In [ ]:
print("Número de horas:", datos.shape[0])
print(f"Demanda media (MW):{datos["demanda_mw"].mean():,.2f} MW")
display(datos.info())

## Ejercicio 3. Gráfica: consumo medio por hora del día
Agrupa por `hora`, calcula la media de `demanda_mw` y dibújala.

**Pista:** `datos.groupby("hora")["demanda_mw"].mean()`

**Pregunta:** ¿a qué hora se consume más? ¿Y menos?

In [ ]:
por_hora = datos.groupby("hora")["demanda_mw"].mean()

plt.plot(por_hora)
plt.xlabel("Hora del día")
plt.ylabel("Demanda media (MW)")
plt.title("Consumo medio por hora")
plt.show()

## Ejercicio 4. Limpiar: ¿faltan datos?
Cuenta los valores vacíos de `demanda_mw` y después borra esas filas.

**Pista:** `.isna().sum()` y `datos.dropna(subset=["columna"])`

In [ ]:
print("Valores vacíos:", datos["demanda_mw"].isna().sum())

# datos = datos.dropna(subset=["demanda_mw"])
print("Horas que quedan:", len(datos))

In [ ]:
print(f"numero de demanda_mw cero o nulo : {datos[((datos["demanda_mw"].isna())|(datos["demanda_mw"] == 0))]["demanda_mw"].shape[0]}")

median_demandaMw = datos["demanda_mw"].median()
median_humedadMadridMw = datos["humedad_madrid"].median()
median_vientoMedioMw = datos["viento_medio"].median()
median_eolicaMw = datos["generacion_eolica_mw"].median()

datos.loc[(datos["demanda_mw"].isna())|(datos["demanda_mw"]== 0),"demanda_mw"] = median_demandaMw
datos.loc[(datos["humedad_madrid"].isna())|(datos["humedad_madrid"] == 0),"humedad_madrid"] = median_humedadMadridMw
datos.loc[(datos["viento_medio"] >= 18),"viento_medio"] = median_vientoMedioMw
datos.loc[(((datos["generacion_eolica_mw"] == 0)| datos["generacion_eolica_mw"].isna()) & (datos["hora"] >= 3)),"generacion_eolica_mw"] = median_eolicaMw

display(datos.info())


## Ejercicio 5. Preparar X e y
- **X**: `mes`, `dia_semana`, `hora`, `fin_de_semana`, `temp_madrid`
- **y**: `demanda_mw`

**Entrenamos con 2015-2017 y examinamos con 2018** (aprendemos del pasado para predecir el futuro).

**Pista:** `datos[datos["anio"] <= 2017]` se queda con las filas hasta 2017.

In [ ]:
entradas = ["mes", "dia_semana", "hora", "fin_de_semana", "temp_madrid"]

entreno = datos[datos["anio"] <= 2017]
prueba = datos[datos["anio"] == 2018]

X_train = entreno[entradas]
y_train = entreno["demanda_mw"]
X_test = prueba[entradas]
y_test = prueba["demanda_mw"]

print("Horas para entrenar:", len(X_train))
print("Horas para examinar:", len(X_test))

## Ejercicio 6. Entrenar una regresión lineal y un Random Forest

**Pista:** se entrena con `modelo.fit(X_train, y_train)`

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lineal = LinearRegression()
lineal.fit(X_train, y_train)

bosque = RandomForestRegressor(n_estimators=100, random_state=42)
bosque.fit(X_train, y_train)

print("Modelos entrenados")

## Ejercicio 7. Métrica: error medio (MAE)
Calcula el MAE de los dos modelos con los datos de 2018.

**Pista:** `mean_absolute_error(y_test, predicciones)`

**Pregunta:** ¿qué modelo se equivoca menos?

In [ ]:
from sklearn.metrics import mean_absolute_error

pred_lineal = lineal.predict(X_test)
pred_bosque = bosque.predict(X_test)

print("Error regresión lineal (MW):", round(mean_absolute_error(y_test, pred_lineal)))
print("Error Random Forest (MW):", round(mean_absolute_error(y_test, pred_bosque)))

## Ejercicio 8. Gráfica: real vs predicho
Dibuja una semana de 2018 (168 horas) con la demanda real y la predicha por el Random Forest.

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(2, 1, 1)
plt.plot(y_test.values[:168], label="Real")
plt.plot(pred_lineal[:168], label="Regresion lineal", linestyle="--", color="r")
plt.plot(pred_bosque[:168], label="Random Forest", linestyle="--")
plt.xlabel("Horas")
plt.ylabel("Demanda (MW)")
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(y_test.values[1000:1168], label="Real", color='b')
plt.plot(pred_lineal[1000:1168], label="Regresion lineal", linestyle="--", color="orange")
plt.plot(pred_bosque[1000:1168], label="Random Forest", linestyle="--", color="y")
plt.xlabel("Horas")
plt.ylabel("Demanda (MW)")
plt.legend()

plt.show()

## Ejercicio 9. Predecir una hora concreta
¿Cuánta electricidad se consumirá un **miércoles de julio a las 17 h con 39 °C en Madrid**?

**Pista:** miércoles es `dia_semana = 2`; no es fin de semana.

In [ ]:
mi_hora = pd.DataFrame([{
    "mes": 7, "dia_semana": 2, "hora": 17,
    "fin_de_semana": 0, "temp_madrid": 39,
}])

print("Demanda estimada:", round(bosque.predict(mi_hora)[0]), "MW")

## Ejercicio 10 (extra). Un domingo de abril a las 4 de la madrugada con 12 °C
Copia el código anterior y cambia los valores. ¿Por qué sale mucho menos?

In [ ]:
mi_hora = pd.DataFrame([{
    "mes": 4, "dia_semana": 8, "hora": 4,
    "fin_de_semana": 1, "temp_madrid": 12,
}])

print("Demanda estimada:", round(bosque.predict(mi_hora)[0]), "MW")